In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")
FIGURES_DIR = os.path.join(OUTPUTS_DIR, "figures")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

train_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "train.csv"), usecols=["full_text", "label"])
val_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "val.csv"), usecols=["full_text", "label"])
test_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "test.csv"), usecols=["full_text", "label"])

df = pd.concat(
    [train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")],
    ignore_index=True
)

df["full_text"] = df["full_text"].fillna("").astype(str).str.lower()

causal_patterns = [
    r"\bbecause\b",
    r"\btherefore\b",
    r"\bdue to\b",
    r"\bas a result\b",
    r"\bleads to\b",
    r"\bled to\b",
    r"\bresults in\b",
    r"\bresulted in\b",
    r"\bcauses\b",
    r"\bcaused by\b",
    r"\bhence\b",
    r"\bthus\b"
]

hedge_patterns = [
    r"\ballegedly\b",
    r"\breportedly\b",
    r"\bmay\b",
    r"\bmight\b",
    r"\bcould\b",
    r"\bpossibly\b",
    r"\bappears\b",
    r"\bseems\b"
]

sensational_patterns = [
    r"\bbreaking\b",
    r"\bunbelievable\b",
    r"\bshocking\b",
    r"\bexclusive\b",
    r"\bmust see\b",
    r"\btruth\b",
    r"\bexposed\b",
    r"\bbombshell\b"
]

def count_patterns(text, patterns):
    total = 0
    for p in patterns:
        total += len(re.findall(p, text))
    return total

df["causal_cues"] = df["full_text"].apply(lambda t: count_patterns(t, causal_patterns))
df["hedge_cues"] = df["full_text"].apply(lambda t: count_patterns(t, hedge_patterns))
df["sensational_cues"] = df["full_text"].apply(lambda t: count_patterns(t, sensational_patterns))

summary = df.groupby(["split", "label"], as_index=False)[["causal_cues", "hedge_cues", "sensational_cues"]].mean()
summary["label_name"] = summary["label"].map({0: "Real(0)", 1: "Fake(1)"})

print(summary)

out_path = os.path.join(TABLES_DIR, "causal_cue_stats.csv")
summary.to_csv(out_path, index=False)
print("\nSaved:", out_path)

test_summary = summary[summary["split"] == "test"].copy()

fig = plt.figure(figsize=(8, 4))
x = np.arange(len(test_summary["label_name"]))
width = 0.25

plt.bar(x - width, test_summary["causal_cues"], width, label="Causal cues")
plt.bar(x, test_summary["hedge_cues"], width, label="Hedging cues")
plt.bar(x + width, test_summary["sensational_cues"], width, label="Sensational cues")
plt.xticks(x, test_summary["label_name"])
plt.ylabel("Average count per article")
plt.title("Cue statistics on Test split")
plt.legend()
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, "cue_stats_test.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved figure:", fig_path)